<a href="https://colab.research.google.com/github/ravindyaparami/Statistical-Learning-e20056/blob/main/E20056_Assignment_7c_Item_Response_Prediction_and_Click_Through_Rate_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


#Answers

In [ ]:
#1
import numpy as np
import plotly.graph_objects as go
# Ability range
theta = np.linspace(-4, 4, 500)

def probability_correct(theta, a, b):
  return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()

# Same discrimination, different difficulties
a1 = 1.0

for b in [-1.0, 0.0, 1.0]:
  fig.add_trace(
      go.Scatter(
          x=theta,
          y=probability_correct(theta, a1, b),
          mode='lines',
          name=f'a={a1}, b={b}'
      )
  )

# A different discrimination value
a2 = 2.0
b2 = 0.0

fig.add_trace(
    go.Scatter(
        x=theta,
        y=probability_correct(theta, a2, b2),
        mode='lines',
        name=f'a={a2}, b={b2}' ) )

fig.update_layout(
    title='2PL Item Response Curves',
    xaxis_title='Ability θ',
    yaxis_title='P(Yᵢ = 1 | Θ = θ)',
    template='plotly_white' )

fig.show()

Interpretation

The difficulty parameter $b_i$ controls the horizontal position of the item response curve.

At

$$
\theta=b_i,
$$

we have

$$
p_i(\theta)=0.5.
$$

Therefore:

Increasing $b_i$ shifts the curve to the right, meaning that a higher ability is required to obtain the same probability of answering correctly.
Decreasing $b_i$ shifts the curve to the left, meaning that the item is easier.
The discrimination parameter $a_i$ controls the steepness of the curve. A larger $a_i$ produces a steeper transition around $\theta=b_i$.
2. Sequential Likelihood Contribution

For a new binary response

$$
y_k\in{0,1},
$$

the likelihood contribution of the $k$-th response is the Bernoulli likelihood
$$
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k},
$$

where
$$
\frac{1}{1+\exp[-a_k(\theta-b_k)]}.
$$

Therefore:

If $y_k=1$,

$$
L(y_k\mid\theta)=p_k(\theta).
$$

If $y_k=0$,

$$
L(y_k\mid\theta)=1-p_k(\theta).
$$

Assuming the responses are conditionally independent given $\Theta=\theta$, the joint likelihood for the running response history

$$
y^{(k)}=(y_1,\ldots,y_k)
$$

is
$$
\prod_{i=1}^{k}
[p_i(\theta)]^{y_i}
[1-p_i(\theta)]^{1-y_i}.
$$

3. Mathematical Formulation of the Running Bayesian Update

At step $k-1$, suppose the current posterior density is

$$
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

This posterior becomes the prior for the next update.

After observing the new response $y_k$, Bayes' theorem gives

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
L(y_k\mid\theta)
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

Substituting the Bernoulli likelihood,

$$
f_{\Theta\mid Y^{(k)}}
\left(
\theta\mid y^{(k)}
\right)
\propto
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
\left(
\theta\mid y^{(k-1)}
\right).
$$

The fully normalized update is
$$
\frac{
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
(\theta\mid y^{(k-1)})
}{
\int
[p_k(u)]^{y_k}
[1-p_k(u)]^{1-y_k}
f_{\Theta\mid Y^{(k-1)}}
(u\mid y^{(k-1)})
,du
}.
$$

Thus, every response modifies the current posterior, and this updated posterior is carried forward to the next question.

4. Dynamic Shifting After a Correct Answer to a Difficult Item

Suppose the user correctly answers a highly difficult item, so

$$
y_k=1
$$

and $b_k$ is large.

For a correct response, the update becomes

$$
f_{\Theta\mid Y^{(k)}}(\theta\mid y^{(k)})
\propto
p_k(\theta)
f_{\Theta\mid Y^{(k-1)}}(\theta\mid y^{(k-1)}).
$$

For a difficult item,
$$
\frac{1}{1+\exp[-a_k(\theta-b_k)]}
$$

is small for low values of $\theta$ and becomes large only for sufficiently high values of $\theta$.

Therefore, multiplying the previous posterior by $p_k(\theta)$:

strongly reduces posterior density at low ability values,
gives relatively more weight to high ability values.

Consequently, the posterior peak generally shifts toward larger values of $\theta$.

The effect can also be seen from the log-likelihood contribution for a correct response:

$$
\log L(y_k=1\mid\theta)=\log p_k(\theta).
$$

Its derivative is
$$
a_k[1-p_k(\theta)]>0.
$$

Thus, a correct answer contributes a positive upward push to the estimated ability.

A correct response to a very difficult item is particularly strong evidence of high ability because such a response would have been unlikely under low values of $\theta$.

5. Effect of the Discrimination Parameter on Posterior Sharpness

The discrimination parameter $a_k$ determines how strongly the probability of a correct response changes with ability.

The slope of the 2PL response function is
$$
a_kp_k(\theta)[1-p_k(\theta)].
$$

The Fisher information supplied by a 2PL item about $\theta$ is
$$
a_k^2p_k(\theta)[1-p_k(\theta)].
$$

Therefore, the information provided by an item increases approximately with $a_k^2$.

When $a_k$ is very large

A large discrimination parameter produces a steep response curve.

The response can strongly distinguish between users whose abilities lie on opposite sides of the item difficulty $b_k$.

If the item is appropriately targeted to the user's ability, the observation contributes substantial information, causing the posterior distribution to become more concentrated or sharper, with a smaller posterior variance.

When $a_k$ is very small

A small discrimination parameter produces a flatter response curve.

The probability of success changes only slowly with $\theta$, so the response contains relatively little information about the user's exact ability.

The posterior changes less after observing the response and generally remains broader, corresponding to greater uncertainty.

The item provides the greatest information around

$$
\theta\approx b_k,
$$

where

$$
p_k(\theta)\approx 0.5.
$$

Thus, both high discrimination and an item difficulty close to the user's current ability are important for reducing posterior uncertainty efficiently.

6. Numerical Implementation Using a Fixed Ability Grid

Because the posterior distribution does not generally have a simple closed-form distribution after applying the logistic likelihood, it can be approximated numerically using a fixed grid.

Step 1: Define a grid

Choose a sufficiently wide range of possible ability values, for example

$$
\theta\in[-4,4].
$$

Represent this range by many equally spaced points:

$$
\theta_1,\theta_2,\ldots,\theta_M.
$$

Step 2: Initialize the prior

Evaluate the standard normal prior at every grid point:
$$
\frac{1}{\sqrt{2\pi}}
\exp\left(-\frac{\theta_j^2}{2}\right).
$$

Normalize numerically so that

$$
\int f^{(0)}(\theta)d\theta=1.
$$

Step 3: Compute the likelihood for the new response

For each grid point, calculate
$$
\frac{1}{1+\exp[-a_k(\theta_j-b_k)]}.
$$

Then calculate
$$
[p_k(\theta_j)]^{y_k}
[1-p_k(\theta_j)]^{1-y_k}.
$$

Step 4: Perform the unnormalized Bayesian update

Multiply the previous posterior by the new likelihood:

L_k(\theta_j)f^{(k-1)}(\theta_j).
$$

Step 5: Normalize computationally

Approximate the normalization constant numerically:
$$
\int \tilde f^{(k)}(\theta)d\theta.
$$

Using a numerical integration method such as the trapezoidal rule,

normalization_constant = np.trapezoid(posterior_unnormalized, theta_grid)

Then normalize:

posterior = posterior_unnormalized / normalization_constant

Hence,
$$
\frac{\tilde f^{(k)}(\theta_j)}
{\int\tilde f^{(k)}(\theta)d\theta}.
$$

This procedure is repeated sequentially after every new item response.

Posterior estimates

The posterior mean is approximated by
$$
E[\Theta\mid y^{(k)}]

\int\theta f^{(k)}(\theta)d\theta.
$$

Numerically,

posterior_mean = np.trapezoid(theta_grid * posterior, theta_grid)

The MAP estimate is
$$
\arg\max_{\theta}f^{(k)}(\theta).
$$

Numerically,

map_estimate = theta_grid[np.argmax(posterior)]
7. Running Simulation for a User with True Ability $\theta_{true}=0.75$

Let

$$
\theta_{true}=0.75
$$

and simulate

$$
n=20
$$

items.

For each item,

$$
b_k\sim\mathcal{N}(0,1)
$$

and

$$
a_k\sim Uniform(0.5,2.0).
$$

The true probability that the user answers item $k$ correctly is
$$
\frac{1}
{1+\exp[-a_k(\theta_{true}-b_k)]}.
$$

Generate

$$
U_k\sim Uniform(0,1).
$$

Then define the simulated response as

$$
y_k=
\begin{cases}
1, & U_k<p_k(\theta_{true}),\
0, & U_k\geq p_k(\theta_{true}).
\end{cases}
$$

The following Python script performs the complete sequential Bayesian simulation and tracks both the posterior mean and MAP estimator.

In [ ]:


import numpy as np
import plotly.graph_objects as go

# -------------------------------------------------
# Settings
# -------------------------------------------------

np.random.seed(42)

theta_true = 0.75
n_items = 20

# Fixed ability grid
theta_grid = np.linspace(-4, 4, 4001)

# -------------------------------------------------
# Initial prior: N(0,1)
# -------------------------------------------------

posterior = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)

# Normalize numerically
posterior = posterior / np.trapezoid(posterior, theta_grid)

# -------------------------------------------------
# Storage
# -------------------------------------------------

steps = [0]

# At step 0, prior mean and prior MAP are both 0
posterior_means = [
    np.trapezoid(theta_grid * posterior, theta_grid)
]

map_estimates = [
    theta_grid[np.argmax(posterior)]
]

responses = []
difficulties = []
discriminations = []
true_probabilities = []

# -------------------------------------------------
# Sequential Bayesian updating
# -------------------------------------------------

for k in range(1, n_items + 1):

    # Random item parameters
    b_k = np.random.normal(0, 1)
    a_k = np.random.uniform(0.5, 2.0)

    # True probability of a correct response
    p_true = 1 / (
        1 + np.exp(-a_k * (theta_true - b_k))
    )

    # Simulate response using U(0,1)
    u = np.random.uniform(0, 1)

    if u < p_true:
        y_k = 1
    else:
        y_k = 0

    # Probability of success over the entire theta grid
    p_grid = 1 / (
        1 + np.exp(-a_k * (theta_grid - b_k))
    )

    # Likelihood contribution
    likelihood = (
        p_grid**y_k
        * (1 - p_grid)**(1 - y_k)
    )

    # Bayesian update
    posterior_unnormalized = posterior * likelihood

    # Sequential normalization
    normalization_constant = np.trapezoid(
        posterior_unnormalized,
        theta_grid
    )

    posterior = (
        posterior_unnormalized
        / normalization_constant
    )

    # Posterior mean
    posterior_mean = np.trapezoid(
        theta_grid * posterior,
        theta_grid
    )

    # MAP estimate
    map_estimate = theta_grid[
        np.argmax(posterior)
    ]

    # Store results
    steps.append(k)
    posterior_means.append(posterior_mean)
    map_estimates.append(map_estimate)

    responses.append(y_k)
    difficulties.append(b_k)
    discriminations.append(a_k)
    true_probabilities.append(p_true)

# -------------------------------------------------
# Display simulated item information
# -------------------------------------------------

print("True ability =", theta_true)
print()

for k in range(n_items):
    print(
        f"Item {k+1:2d}: "
        f"a={discriminations[k]:.3f}, "
        f"b={difficulties[k]:.3f}, "
        f"P(correct)={true_probabilities[k]:.3f}, "
        f"response={responses[k]}"
    )

# -------------------------------------------------
# Plot estimator progression
# -------------------------------------------------

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=steps,
        y=posterior_means,
        mode='lines+markers',
        name='Posterior Mean'
    )
)

fig.add_trace(
    go.Scatter(
        x=steps,
        y=map_estimates,
        mode='lines+markers',
        name='MAP Estimate'
    )
)

# True ability reference line
fig.add_hline(
    y=theta_true,
    line_dash='dash',
    annotation_text='True Ability θ = 0.75',
    annotation_position='top left'
)

fig.update_layout(
    title='Sequential Bayesian Estimation of User Ability',
    xaxis_title='Number of Observed Items',
    yaxis_title='Estimated Ability θ',
    template='plotly_white',
    legend_title='Estimator'
)

fig.show()

True ability = 0.75

Item  1: a=1.598, b=0.497, P(correct)=0.600, response=1
Item  2: a=0.734, b=-0.138, P(correct)=0.657, response=1
Item  3: a=0.531, b=1.579, P(correct)=0.392, response=0
Item  4: a=1.749, b=0.767, P(correct)=0.492, response=1
Item  5: a=0.956, b=-0.463, P(correct)=0.761, response=1
Item  6: a=1.148, b=-0.466, P(correct)=0.801, response=1
Item  7: a=0.938, b=-1.013, P(correct)=0.839, response=1
Item  8: a=1.184, b=0.314, P(correct)=0.626, response=0
Item  9: a=1.389, b=0.068, P(correct)=0.721, response=1
Item 10: a=1.411, b=-1.425, P(correct)=0.956, response=1
Item 11: a=1.526, b=-0.601, P(correct)=0.887, response=1
Item 12: a=0.683, b=-0.292, P(correct)=0.671, response=1
Item 13: a=0.968, b=0.823, P(correct)=0.482, response=0
Item 14: a=1.320, b=-1.221, P(correct)=0.931, response=1
Item 15: a=0.633, b=0.738, P(correct)=0.502, response=1
Item 16: a=0.568, b=0.171, P(correct)=0.581, response=1
Item 17: a=1.743, b=-1.479, P(correct)=0.980, response=1
Item 18: a=0.921, 

8. Analysis of Convergence Over Time

At step $0$, no responses have been observed, so the ability estimate is determined entirely by the prior

$$
\Theta\sim\mathcal{N}(0,1).
$$

Hence, both the prior mean and prior MAP estimate are approximately

$$
\hat{\theta}^{(0)}=0.
$$

The true ability is

$$
\theta_{true}=0.75.
$$

As more responses are observed, each response contributes new evidence through the likelihood

$$
[p_k(\theta)]^{y_k}
[1-p_k(\theta)]^{1-y_k}.
$$

The posterior therefore gradually shifts from being dominated by the prior toward being dominated by the observed response data.

In general, as $k$ increases:

The posterior mean and MAP estimates tend to move toward the true ability $\theta_{true}=0.75$.
The estimates may fluctuate, particularly during the early stages, because individual correct or incorrect responses are random.
A surprising response can temporarily move the estimate away from the true value. For example, an incorrect answer to an easy item may reduce the estimated ability, while a correct answer to a difficult item may increase it substantially.
As more informative items are observed, the influence of any single response becomes smaller because the posterior already contains information accumulated from previous responses.
The posterior distribution generally becomes narrower as information accumulates. This represents a reduction in posterior uncertainty.

Therefore, convergence of the posterior mean and MAP toward the true value, together with increasing sharpness of the posterior distribution, indicates that the platform is becoming more confident and more precise in its measurement of the user's latent ability.

However, because only $20$ items are used and the responses are stochastic, the estimates are not guaranteed to equal exactly

$$
\theta_{true}=0.75.
$$

Their convergence also depends on the informativeness of the selected items. Items with high discrimination and difficulties close to the user's true ability generally provide more information and lead to faster reduction in uncertainty.

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

#Answers

## 1. Structural Probability and Properties

The Beta distribution is controlled by two shape parameters, $\alpha$ and $\beta$, which determine the concentration of the probability mass.

In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

# Define domain
theta = np.linspace(0, 1, 500)

# Define parameter pairs
params = [
    {"alpha": 1, "beta": 1, "label": "Uninformative (1, 1)", "color": "blue"},
    {"alpha": 2, "beta": 8, "label": "Right-Skewed (2, 8)", "color": "orange"},
    {"alpha": 8, "beta": 2, "label": "Left-Skewed (8, 2)", "color": "green"}
]

fig = go.Figure()

for p in params:
    pdf_values = beta.pdf(theta, p["alpha"], p["beta"])
    fig.add_trace(go.Scatter(
        x=theta, y=pdf_values, mode='lines',
        name=p["label"], line=dict(color=p["color"], width=2)
    ))

fig.update_layout(
    title="Beta Distribution P.D.F for Different Shape Parameters",
    xaxis_title="Theta (Conversion Rate)",
    yaxis_title="Density",
    template="plotly_white"
)
fig.show()

**Interpretation of Shifting Mass:**
*   **Uninformative $(\alpha=1, \beta=1)$:** The density is uniform across $[0, 1]$. All values of $\Theta$ are equally likely.
*   **Right-Skewed $(\alpha=2, \beta=8)$:** The mass shifts toward $0$. The expected value is $2/10 = 0.2$. This represents a strong prior belief that the advertisement will have a low CTR.
*   **Left-Skewed $(\alpha=8, \beta=2)$:** The mass shifts toward $1$. The expected value is $8/10 = 0.8$. This represents a strong prior belief that the advertisement will perform exceptionally well.

Increasing $\alpha$ relative to $\beta$ pulls the center of mass toward $1$, while increasing $\beta$ relative to $\alpha$ pushes it toward $0$.

---

## 2. Sequential Likelihood and Joint History

For a single Bernoulli trial at step $k$, the likelihood of observing response $y_k \in \{0, 1\}$ given the true rate $\theta$ is:

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

For the running history vector $\mathbf{y}^{(k)}$, assuming conditional independence of impressions, the joint likelihood is the product of the individual likelihoods:

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^k \theta^{y_i} (1 - \theta)^{1 - y_i} = \theta^{\sum_{i=1}^k y_i} (1 - \theta)^{k - \sum_{i=1}^k y_i}$$

---

## 3. Closed-Form Analytical Updates (Conjugacy)

By Bayes' Theorem, the posterior at step $k$ is proportional to the prior at step $k-1$ multiplied by the likelihood of the new observation $y_k$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \times L(y_k \mid \theta)$$

Substituting the Beta density for the prior and the Bernoulli likelihood:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] \times \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right]$$

Combine the exponents for $\theta$ and $(1 - \theta)$:

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$

This kernel matches the functional form of a new Beta distribution. Therefore, the posterior is exactly $\text{Beta}(\alpha_k, \beta_k)$, where the parameters update via simple addition:

$$\alpha_k = \alpha_{k-1} + y_k$$
$$\beta_k = \beta_{k-1} + (1 - y_k)$$

Because the posterior takes the exact same parametric form as the prior, the Beta distribution is the **conjugate prior** for the Binomial/Bernoulli likelihood. The Posterior Mean at step $k$ is exactly:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

---

## 4. Dynamic Shifting Mechanics

Because of the additive updates:
*   **Observing a click ($y_k = 1$):** Increases $\alpha_k$ by 1, leaving $\beta_k$ unchanged. This analytically shifts the center of mass to the right, reflecting increased optimism about the CTR.
*   **Observing a non-click ($y_k = 0$):** Increases $\beta_k$ by 1, leaving $\alpha_k$ unchanged. This analytically shifts the center of mass to the left, reflecting pessimism.

**Contrast with Non-Conjugate Setups (e.g., 2PL IRT):**
In conjugate models, updating the posterior requires zero calculus — it is a $O(1)$ arithmetic operation tracking two integers. The normalizing constant $\mathrm{B}(\alpha_k, \beta_k)$ is known implicitly. In non-conjugate models like the 2-Parameter Logistic (2PL) Item Response Theory model, multiplying a normal prior by a logistic likelihood produces an unrecognizable posterior kernel. Because there is no closed-form algebraic rule, you must use numerical grid integration (quadrature) or Markov Chain Monte Carlo (MCMC) to evaluate the normalizing integral at every step, which is computationally expensive for real-time streaming data.

---

## 5. Running Point Estimators

Directly from the updated parameters $\alpha_k$ and $\beta_k$, the point estimators at step $k$ are:

*   **Running Posterior Mean:**
    $$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$$
*   **Running Maximum A Posteriori (MAP):**
    $$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2} \quad \text{for } \alpha_k, \beta_k > 1$$
    *(Note: If $\alpha_k, \beta_k \le 1$, the MAP estimate sits at the boundaries or is undefined).*

---

## 6. Performance Tracking and Convergence Analysis

In [2]:
import numpy as np
import plotly.graph_objects as go

# Parameters
n_impressions = 100
theta_true = 0.35
alpha_0, beta_0 = 1, 1

# Simulation arrays
clicks = np.random.rand(n_impressions) < theta_true
clicks = clicks.astype(int)

bayes_estimates = []
map_estimates = []

alpha_k, beta_k = alpha_0, beta_0

for y_k in clicks:
    # Update shape parameters
    alpha_k += y_k
    beta_k += (1 - y_k)

    # Calculate estimators
    bayes_est = alpha_k / (alpha_k + beta_k)

    # MAP is bounded to [0,1], undefined for alpha, beta < 1. Handle edge cases safely:
    if alpha_k + beta_k > 2:
        map_est = max(0.0, min(1.0, (alpha_k - 1) / (alpha_k + beta_k - 2)))
    else:
        map_est = np.nan # Undefined at early extreme steps if uniform prior

    bayes_estimates.append(bayes_est)
    map_estimates.append(map_est)

# Visualization
steps = np.arange(1, n_impressions + 1)
fig = go.Figure()

fig.add_trace(go.Scatter(x=steps, y=bayes_estimates, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(x=steps, y=map_estimates, mode='lines', name='MAP Estimate'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True CTR (0.35)")

fig.update_layout(
    title="Sequential Bayesian Estimation of CTR over 100 Impressions",
    xaxis_title="Impression Step (k)",
    yaxis_title="Estimated CTR (Theta)",
    template="plotly_white",
    yaxis=dict(range=[0, 1])
)
fig.show()

**Analysis of Convergence:**

As the sampling size $k$ approaches 100, the distance between both estimators ($\widehat{\theta}_{\mathrm{Bayes}}$ and $\widehat{\theta}_{\mathrm{MAP}}$) and the true parameter $\theta_{\text{true}}$ shrinks. Early on (e.g., $k < 20$), the estimators exhibit high variance and are highly sensitive to the initial prior and early streak luck (e.g., three clicks in a row). By the Bernstein-von Mises theorem, as $k \to \infty$, the likelihood term overwhelmingly dominates the initial prior $(\alpha_0, \beta_0)$. The posterior distribution tightens around the Maximum Likelihood Estimate, ensuring that regardless of the starting point, the Bayesian estimates logically converge to the true data-generating rate of 0.35.